<a href="https://colab.research.google.com/github/srilakshmi-saladi/unet/blob/main/brain_effecientnet_densenet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q timm

import os, random, warnings, copy
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, cohen_kappa_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
import timm

warnings.filterwarnings("ignore")

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')

Device: cpu
Mounted at /content/drive


In [ ]:
BASE_DIR = Path('/content/drive/MyDrive/brain dataset/brisc2025/classification_task')
IMG_SIZE = 224
BATCH_SIZE = 32

def image_files_in(folder):
    return [p for p in folder.rglob('*') if p.suffix.lower() in ['.png', '.jpg', '.jpeg']]

train_dir = BASE_DIR / 'train'
val_dir = BASE_DIR / 'val'
test_dir = BASE_DIR / 'test'
records = []

if train_dir.exists() and test_dir.exists():
    class_names = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
    class2idx = {c:i for i,c in enumerate(class_names)}
    for split_name, split_dir in [('train', train_dir), ('val', val_dir), ('test', test_dir)]:
        if split_dir.exists():
            for cls in class_names:
                for p in image_files_in(split_dir / cls):
                    records.append({'path': str(p), 'label_name': cls, 'label': class2idx[cls], 'split': split_name})
    df = pd.DataFrame(records)
else:
    class_names = sorted([d.name for d in BASE_DIR.iterdir() if d.is_dir()])
    class2idx = {c:i for i,c in enumerate(class_names)}
    for cls in class_names:
        for p in image_files_in(BASE_DIR / cls):
            records.append({'path': str(p), 'label_name': cls, 'label': class2idx[cls]})
    df = pd.DataFrame(records)
    train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
    val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED)
    train_df['split'] = 'train'
    val_df['split'] = 'val'
    test_df['split'] = 'test'
    df = pd.concat([train_df, val_df, test_df], ignore_index=True)

train_df = df[df['split']=='train'].reset_index(drop=True)
val_df = df[df['split']=='val'].reset_index(drop=True)
test_df = df[df['split']=='test'].reset_index(drop=True)

# THE VALIDATION FIX
if len(val_df) == 0:
    print("[INFO] No validation folder found! Splitting 15% of train data for validation...")
    train_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df['label'], random_state=SEED)
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

print("\nClasses found:", class_names)
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

def get_transforms(img_size, mode='train'):
    if mode == 'train':
        return T.Compose([
            T.Resize((img_size + 16, img_size + 16)),
            T.RandomCrop(img_size),
            T.RandomHorizontalFlip(),
            T.RandomRotation(15),
            T.ColorJitter(brightness=0.1, contrast=0.1),
            T.ToTensor(),
            T.Normalize(MEAN, STD),
        ])
    return T.Compose([
        T.Resize((img_size, img_size)),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])

class ImageClsDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        y = int(row['label'])
        if self.transform: img = self.transform(img)
        return img, y

counts = train_df['label'].value_counts().sort_index().values.astype(float)
class_weights = (1.0 / counts)
class_weights = class_weights / class_weights.sum() * len(class_names)
CLASS_WEIGHTS_TENSOR = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

NUM_WORKERS = 2
train_loader = DataLoader(ImageClsDataset(train_df, get_transforms(IMG_SIZE, 'train')), batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=2)
val_loader = DataLoader(ImageClsDataset(val_df, get_transforms(IMG_SIZE, 'val')), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=2)
test_loader = DataLoader(ImageClsDataset(test_df, get_transforms(IMG_SIZE, 'val')), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, prefetch_factor=2)

[INFO] No validation folder found! Splitting 15% of train data for validation...

Classes found: ['glioma', 'meningioma', 'no_tumor', 'pituitary']
Train size: 4250, Val size: 750, Test size: 1000


In [ ]:
# 1. EfficientNet-B0
def build_efficientnet(num_classes):
    return timm.create_model('efficientnet_b0', pretrained=True, num_classes=num_classes, drop_rate=0.3)

# 2. DenseNet-121
def build_densenet(num_classes):
    return timm.create_model('densenet121', pretrained=True, num_classes=num_classes, drop_rate=0.3)

In [ ]:
def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

def mixup_loss(criterion, out, ya, yb, lam):
    return lam * criterion(out.float(), ya) + (1-lam) * criterion(out.float(), yb)

def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in tqdm(loader, leave=False, desc='Train'):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            if random.random() < 0.5:
                imgs, ya, yb, lam = mixup_data(imgs, labels)
                out = model(imgs)
                loss = mixup_loss(criterion, out, ya, yb, lam)
            else:
                out = model(imgs)
                loss = criterion(out.float(), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    return total_loss/total, correct/total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    criterion_eval = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS_TENSOR)
    total_loss, all_preds, all_labels = 0.0, [], []
    total = 0
    for imgs, labels in tqdm(loader, leave=False, desc='Eval'):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast():
            out = model(imgs)
        loss = criterion_eval(out.float(), labels)
        probs = F.softmax(out, dim=1)
        total_loss += loss.item() * imgs.size(0)
        all_preds.extend(probs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        total += imgs.size(0)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    return total_loss/total, acc, f1, qwk, np.array(all_preds), np.array(all_labels)

torch.backends.cudnn.benchmark = True

In [ ]:
EPOCHS = 12
LR = 3e-4

models_to_test = {
    "EfficientNet-B0": build_efficientnet(len(class_names)),
    "DenseNet-121": build_densenet(len(class_names))
}

best_states = {}
model_histories = {}

for model_name, model in models_to_test.items():
    print(f"\n{'='*40}")
    print(f"🚀 STARTING TRAINING FOR: {model_name}")
    print(f"{'='*40}")

    model = model.to(DEVICE)
    if torch.__version__ >= "2.0.0":
        import torch._dynamo
        torch._dynamo.config.suppress_errors = True
        model = torch.compile(model)

    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS_TENSOR, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
    scaler = GradScaler()

    best_f1 = -1
    best_state = None
    history = []

    for ep in range(1, EPOCHS+1):
        tl, ta = train_epoch(model, train_loader, optimizer, criterion, scaler)
        vl, va, vf1, vqwk, _, _ = evaluate(model, val_loader)
        scheduler.step()
        history.append([ep, tl, ta, vl, va, vf1, vqwk])
        print(f"[{model_name}] Ep {ep:02d}/{EPOCHS} | Train Acc: {ta:.4f} | Val Acc: {va:.4f} | Val F1: {vf1:.4f}")

        if vf1 > best_f1:
            best_f1 = vf1
            best_state = copy.deepcopy(model.state_dict())

    # Save the best version of this model
    best_states[model_name] = best_state
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), f'/content/{model_name}_best.pth')
    model_histories[model_name] = history
    print(f"✅ Finished {model_name}. Best Val F1: {best_f1:.4f}")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/32.3M [00:00<?, ?B/s]


🚀 STARTING TRAINING FOR: EfficientNet-B0


[EfficientNet-B0] Ep 01/12 | Train Acc: 0.6451 | Val Acc: 0.8627 | Val F1: 0.8651


[EfficientNet-B0] Ep 02/12 | Train Acc: 0.7618 | Val Acc: 0.9360 | Val F1: 0.9350


...

[EfficientNet-B0] Ep 09/12 | Train Acc: 0.8130 | Val Acc: 0.9920 | Val F1: 0.9926


[EfficientNet-B0] Ep 10/12 | Train Acc: 0.8047 | Val Acc: 0.9893 | Val F1: 0.9900


[EfficientNet-B0] Ep 11/12 | Train Acc: 0.8250 | Val Acc: 0.9920 | Val F1: 0.9925


[EfficientNet-B0] Ep 12/12 | Train Acc: 0.7995 | Val Acc: 0.9920 | Val F1: 0.9926
✅ Finished EfficientNet-B0. Best Val F1: 0.9926

🚀 STARTING TRAINING FOR: DenseNet-121


Train:  42%|████▏     | 55/132 [2:26:24<3:17:46, 154.11s/it]

In [1]:
# TTA Dataset Setup
tta_transforms = [
    T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)]),
    T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(p=1), T.ToTensor(), T.Normalize(MEAN, STD)]),
]

class TTADataset(Dataset):
    def __init__(self, df, transforms_list):
        self.df = df.reset_index(drop=True)
        self.transforms_list = transforms_list
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        label = int(row['label'])
        imgs = [tr(img) for tr in self.transforms_list]
        return imgs, label

tta_dataset = TTADataset(test_df, tta_transforms)

@torch.no_grad()
def evaluate_tta(model, tta_dataset, batch_size=8):
    model.eval()
    all_probs, all_labels = [], []
    for start in tqdm(range(0, len(tta_dataset), batch_size), desc='TTA Eval'):
        batch_items = [tta_dataset[i] for i in range(start, min(start+batch_size, len(tta_dataset)))]
        labels = torch.tensor([x[1] for x in batch_items]).to(DEVICE)
        probs_sum = None
        for t_idx in range(len(tta_transforms)):
            imgs = torch.stack([x[0][t_idx] for x in batch_items]).to(DEVICE)
            with autocast():
                out = model(imgs)
                probs = F.softmax(out, dim=1)
            probs_sum = probs if probs_sum is None else probs_sum + probs
        probs_avg = probs_sum / len(tta_transforms)
        all_probs.extend(probs_avg.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    preds = np.argmax(all_probs, axis=1)
    acc = accuracy_score(all_labels, preds)
    f1 = f1_score(all_labels, preds, average='macro', zero_division=0)
    qwk = cohen_kappa_score(all_labels, preds, weights='quadratic')
    return acc, f1, qwk

# Run Final Evaluation for all models
results = []

print("\n" + "="*50)
print("🏆 FINAL MODEL COMPARISON ON UNSEEN TEST DATA")
print("="*50)

for model_name, state_dict in best_states.items():
    print(f"\nEvaluating {model_name}...")

    # Reload model framework
    if model_name == "EfficientNet-B0":
        model = build_efficientnet(len(class_names)).to(DEVICE)
    elif model_name == "DenseNet-121":
        model = build_densenet(len(class_names)).to(DEVICE)

    model.load_state_dict(state_dict)

    acc, f1, qwk = evaluate_tta(model, tta_dataset, batch_size=8)
    results.append({
        "Model": model_name,
        "Accuracy": f"{acc*100:.2f}%",
        "Macro F1": f"{f1*100:.2f}%",
        "QWK Score": f"{qwk*100:.2f}%"
    })

# Print beautiful final table
results_df = pd.DataFrame(results)
display(results_df)

# Find the winner
winner = results_df.loc[results_df["Accuracy"].str.rstrip('%').astype(float).idxmax()]
print(f"\n👑 THE WINNER IS: {winner['Model']} with {winner['Accuracy']} Accuracy!")

NameError: name 'T' is not defined